# Midpoint placebo experiment

This notebook places placebo cutoffs midway between adjacent real cutoffs.

In [ ]:
from pathlib import Path
from math import comb
import sys


def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cp_lfdr").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root containing src/cp_lfdr."
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SOURCE_DIR = PROJECT_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
PAPER_ROOT = PROJECT_ROOT.parent
PAPER_IMAGE_DIR = PAPER_ROOT / "images"
SAVE_FIGURES = True
SAVE_TEX = True
MIRROR_TO_PAPER_IMAGES = (PAPER_ROOT / "cp-lfdr-project.tex").is_file()
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
if MIRROR_TO_PAPER_IMAGES:
    PAPER_IMAGE_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, stem):
    """Save PDF, PNG, and optionally PGF-backed TeX copies."""
    if not SAVE_FIGURES:
        return []
    destinations = [FIGURE_DIR]
    if MIRROR_TO_PAPER_IMAGES:
        destinations.append(PAPER_IMAGE_DIR)
    paths = []
    for destination in destinations:
        pdf_path = destination / f"{stem}.pdf"
        png_path = destination / f"{stem}.png"
        fig.savefig(pdf_path, bbox_inches="tight")
        fig.savefig(png_path, bbox_inches="tight", dpi=300)
        paths.extend([pdf_path, png_path])
        if SAVE_TEX:
            tex_path = destination / f"{stem}.tex"
            fig.savefig(tex_path, format="pgf", bbox_inches="tight")
            paths.append(tex_path)
    print("Saved figure:", ", ".join(str(path) for path in paths))
    return paths

SEED = 0
ALPHA = 0.2
DCP_DRAWS = comb(8, 4)
NUMERICAL_TOLERANCE = 1e-8
MINIMUM_SIDE_COUNT = 4
MAXIMUM_TOTAL_COUNT = 20
LOCAL_RD_DISPLAY_WINDOW = 0.5
BANDWIDTHS = [
    0.01, 0.02, 0.03, 0.04, 0.05,
    0.10, 0.15, 0.20, 0.25,
]
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import display
from rdrobust import rdrobust

get_ipython().run_line_magic("matplotlib", "inline")
ALPHA_GRID = np.arange(0, 0.301, 0.001)

from cp_lfdr.multiple_testing import sl_procedure
from cp_lfdr.rd import (
    compute_pvalues_window,
    construct_year_specific_midpoint_base,
)

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
YEAR_COLORS = {1: "#4C72B0", 2: "#55A868", 3: "#C44E52"}
PLACEBO_COLOR = "#8172B2"
PLACEBO_PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2",
    "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD",
]

# Step 1: construct and match midpoint placebos within each town-year

Within each town-year, real cutoffs are ordered by their recovered score
positions and a placebo is placed at the midpoint of each adjacent gap.
Across cohorts, observations are pooled only when the same two `z` labels bound
the construction. Each cohort retains its own midpoint location.

In [ ]:
rd_path = DATA_DIR / "raw" / "rd" / "data-AER-3.dta"
if not rd_path.exists():
    raise FileNotFoundError(
        f"Missing {rd_path}. Follow data/README.md before running."
    )

raw_rd_data = pd.read_stata(
    rd_path, columns=["dzag", "bcg", "uazY", "sid2"]
)
identifier_parts = raw_rd_data["uazY"].str.extract(
    r"(\d+)z(\d+)Y(\d+)"
)
identifier_parts.columns = ["town", "z", "Y"]
identifier_parts = identifier_parts.astype(int)
raw_rd_data[["town", "z", "Y"]] = identifier_parts
raw_rd_data["dzag"] = (
    raw_rd_data["dzag"].astype(float).round(2)
)
rd_data = raw_rd_data.dropna(subset=["bcg"]).copy()

cutoff_table, year_gap_table, midpoint_placebo_data = (
    construct_year_specific_midpoint_base(rd_data)
)

# The same construction cannot appear twice within one town-year.
assert not year_gap_table.duplicated(
    ["town", "Y", "z_low", "z_high"]
).any()
assert not midpoint_placebo_data.duplicated(
    ["placebo_id", "Y", "sid2"]
).any()

numeric_z_order = cutoff_table.sort_values(["town", "Y", "z"])
numeric_z_order["cutoff_increment"] = (
    numeric_z_order.groupby(["town", "Y"])["cutoff_position"].diff()
)
nonincreasing_numeric_z = int(
    (numeric_z_order["cutoff_increment"].dropna() <= 0).sum()
)
construction_audit = (
    year_gap_table.groupby("placebo_id", as_index=False)
    .agg(
        town=("town", "first"),
        z_low=("z_low", "first"),
        z_high=("z_high", "first"),
        available_years=("Y", "nunique"),
        distinct_gap_ranks=("gap_rank", "nunique"),
    )
)
audit = pd.Series({
    "towns": rd_data["town"].nunique(),
    "town-years": cutoff_table[["town", "Y"]].drop_duplicates().shape[0],
    "year-specific real cutoffs": len(cutoff_table),
    "year-specific candidate constructions": len(year_gap_table),
    "unique pooled placebo constructions": len(construction_audit),
    "constructions available in multiple years": int(
        (construction_audit["available_years"] > 1).sum()
    ),
    "constructions whose ordinal rank changes across years": int(
        (construction_audit["distinct_gap_ranks"] > 1).sum()
    ),
    "zero-width adjacent gaps": int((year_gap_table["gap_width"] == 0).sum()),
    "nonincreasing numeric-z steps": nonincreasing_numeric_z,
    "student-construction rows": len(midpoint_placebo_data),
})
display(audit.to_frame("count"))

print("Student-level and cutoff-table gap widths agree exactly.")
display(
    year_gap_table.sort_values(
        ["town", "Y", "left_cutoff_position"]
    ).head(12)
)

## Visual audit of cutoff geometry

At $h=0.05$, each placebo window must lie strictly between its neighboring real
cutoffs and pass the pooled $4/4/20$ sample-size filter. Exact midpoint
observations are removed.

Each retained town is shown in three cohort panels. Town 20787 is a particularly
clear illustration for the paper. Letters and colors track the same construction
across cohorts; shaded regions show the $h=0.05$ windows.

In [ ]:
def filter_midpoint_placebo_design(
    placebo_data,
    *,
    bandwidth,
    level="placebo_id",
    distance_column="placebo_distance",
    year_column="Y",
    minimum_per_side=4,
    maximum_total=20,
    numerical_tolerance=1e-8,
):
    """Apply contamination, support, and pooled sample-size restrictions."""
    safe = placebo_data[
        placebo_data["gap_width"]
        > 2 * bandwidth + numerical_tolerance
    ].copy()

    # Mirror the main RD support check before restricting to the local window.
    support = (
        safe.groupby(level, as_index=False)[distance_column]
        .agg(_minimum_distance="min", _maximum_distance="max")
    )
    supported = support[
        (support["_minimum_distance"] <= -bandwidth + numerical_tolerance)
        & (support["_maximum_distance"] >= bandwidth - numerical_tolerance)
    ][[level]]
    safe = safe.merge(
        supported, on=level, how="inner", validate="many_to_one"
    )

    window = safe[
        (safe[distance_column].abs() <= bandwidth + numerical_tolerance)
        & (safe[distance_column].abs() > numerical_tolerance)
    ].copy()
    window["_is_left"] = window[distance_column] < 0
    window["_is_right"] = window[distance_column] > 0
    counts = (
        window.groupby(level, as_index=False)
        .agg(
            n_left=("_is_left", "sum"),
            n_right=("_is_right", "sum"),
            n_total=(distance_column, "size"),
            contributing_years=(year_column, "nunique"),
        )
    )
    retained = counts[
        (counts["n_left"] >= minimum_per_side)
        & (counts["n_right"] >= minimum_per_side)
        & (counts["n_total"] <= maximum_total)
    ].copy()
    sample = window.merge(
        retained[[level]], on=level, how="inner", validate="many_to_one"
    )
    return retained.reset_index(drop=True), sample


BASELINE_BANDWIDTH = 0.05
baseline_placebos, baseline_sample = filter_midpoint_placebo_design(
    midpoint_placebo_data,
    bandwidth=BASELINE_BANDWIDTH,
    minimum_per_side=MINIMUM_SIDE_COUNT,
    maximum_total=MAXIMUM_TOTAL_COUNT,
)
safe_baseline_year_gaps = year_gap_table[
    year_gap_table["gap_width"]
    > 2 * BASELINE_BANDWIDTH + NUMERICAL_TOLERANCE
]
print(
    f"At h={BASELINE_BANDWIDTH:.2f}: "
    f"{len(safe_baseline_year_gaps):,} valid cohort-specific constructions "
    f"and {len(baseline_placebos):,} testable pooled placebo constructions "
    f"across {baseline_sample['town'].nunique():,} towns."
)
display(baseline_placebos.head())

visual_cutoffs = cutoff_table.copy()
visual_scores = rd_data.merge(
    visual_cutoffs[["town", "Y", "z", "cutoff_position"]],
    on=["town", "Y", "z"],
    how="left",
    validate="many_to_one",
)
visual_scores["admission_score"] = (
    visual_scores["dzag"] + visual_scores["cutoff_position"]
).round(6)
visual_scores = (
    visual_scores.groupby(["town", "Y", "sid2"], as_index=False)
    .agg(admission_score=("admission_score", "median"), bcg=("bcg", "first"))
)
contributing_construction_years = baseline_sample[
    ["town", "Y", "z_low", "z_high", "placebo_id"]
].drop_duplicates()
visual_placebos = year_gap_table.merge(
    contributing_construction_years,
    on=["town", "Y", "z_low", "z_high", "placebo_id"],
    how="inner",
    validate="one_to_one",
)

# Show every town containing at least one retained placebo construction.
PAPER_EXAMPLE_TOWN = 20787
town_ids = sorted(visual_placebos["town"].unique())
town_ids = [PAPER_EXAMPLE_TOWN] + [
    town for town in town_ids if town != PAPER_EXAMPLE_TOWN
]
gallery_legend = [
    Line2D([0], [0], marker="o", color="none",
           markerfacecolor="#aeb4b7", markeredgecolor="none",
           markersize=5, label="Students"),
    Line2D([0], [0], color="#777777", linestyle="-",
           linewidth=1.0, label="Real cutoff"),
    Line2D([0], [0], color=PLACEBO_COLOR, linestyle="--",
           linewidth=1.8, label="Placebo cutoff"),
    Line2D([0], [0], color=PLACEBO_COLOR, linewidth=8, alpha=0.15,
           label=rf"$h={BASELINE_BANDWIDTH:.2f}$ window"),
]

for town in town_ids:
    town_students = visual_scores[visual_scores["town"] == town]
    town_cutoffs = visual_cutoffs[visual_cutoffs["town"] == town]
    town_placebos = visual_placebos[visual_placebos["town"] == town]
    placebo_order = (
        town_placebos.groupby("placebo_id")["midpoint_position"]
        .mean().sort_values().index.tolist()
    )
    placebo_labels = {
        placebo_id: (chr(65 + index) if index < 26 else str(index + 1))
        for index, placebo_id in enumerate(placebo_order)
    }
    placebo_colors = {
        placebo_id: PLACEBO_PALETTE[index % len(PLACEBO_PALETTE)]
        for index, placebo_id in enumerate(placebo_order)
    }
    x_min = town_placebos["left_cutoff_position"].min()
    x_max = town_placebos["right_cutoff_position"].max()
    x_margin = max(0.15, 0.08 * (x_max - x_min))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True,
                             sharey=True, squeeze=False)
    for ax, year in zip(axes.flat, sorted(YEAR_COLORS)):
        cohort = town_students[town_students["Y"] == year]
        cohort_cutoffs = town_cutoffs[town_cutoffs["Y"] == year]
        cohort_placebos = town_placebos[town_placebos["Y"] == year]
        ax.scatter(cohort["admission_score"], cohort["bcg"],
                   s=10, alpha=0.32, color="#aeb4b7", edgecolors="none",
                   rasterized=True)
        for cutoff in cohort_cutoffs["cutoff_position"]:
            ax.axvline(cutoff, color="#777777", linewidth=0.9, alpha=0.45)
        for construction in cohort_placebos.itertuples(index=False):
            midpoint = construction.midpoint_position
            color = placebo_colors[construction.placebo_id]
            label = placebo_labels[construction.placebo_id]
            ax.axvspan(midpoint - BASELINE_BANDWIDTH,
                       midpoint + BASELINE_BANDWIDTH,
                       color=color, alpha=0.10, linewidth=0)
            ax.axvline(midpoint, color=color, linestyle="--",
                       linewidth=1.8, alpha=0.98)
            ax.text(midpoint, 10.08, label, color=color, fontsize=8,
                    fontweight="bold", ha="center", va="top",
                    clip_on=True)
        ax.set_title(
            f"Cohort {year}: {len(cohort_placebos)} cutoff "
            f"location{'s' if len(cohort_placebos) != 1 else ''}"
        )
        ax.set_xlabel("Median-centered Admission Score")
        ax.set_ylim(5.0, 10.15)
        ax.set_xlim(x_min - x_margin, x_max + x_margin)
        ax.grid(True, axis="y", linestyle=":", linewidth=0.6, alpha=0.3,
                color="#cccccc")
    axes.flat[0].set_ylabel("Baccalaureate Grade")
    fig.text(
        0.01, 0.975,
        f"Town {town}: {len(placebo_order)} tested placebo "
        f"construction{'s' if len(placebo_order) != 1 else ''}",
        ha="left", va="top", fontsize=13,
    )
    fig.legend(handles=gallery_legend, frameon=False, ncol=4,
               loc="upper right", bbox_to_anchor=(0.99, 0.99))
    fig.tight_layout(rect=[0, 0, 1, 0.88])
    if town == PAPER_EXAMPLE_TOWN:
        _ = save_figure(fig, "rd_placebo_town_20787")
    display(fig)
    plt.close(fig)

# Step 2: rejection results across bandwidths

For each bandwidth, the placebo family is reconstructed and the geometry and
$4/4/20$ filters are reapplied. Ordinary and Compound p-values are exact;
Density-Compound uses 70 seed-0 draws, matching the main analysis.

In [ ]:
def separately_pooled_exact_pvalues(exact_statistics, observed_statistics):
    level_ids = list(observed_statistics)
    observed = np.asarray([observed_statistics[i] for i in level_ids])
    tails = np.empty((len(level_ids), len(level_ids)), dtype=float)
    for row, source_id in enumerate(level_ids):
        distribution = np.sort(np.asarray(exact_statistics[source_id]))
        first = np.searchsorted(distribution, observed, side="left")
        tails[row] = (len(distribution) - first) / len(distribution)
    return pd.Series(tails.mean(axis=0), index=level_ids,
                     name="exact_compound_p")


def compute_midpoint_results(sample, bandwidth):
    output = compute_pvalues_window(
        sample, "outcome", level="placebo_id", alpha=ALPHA,
        test_stat="welch_p", dcp_draws=DCP_DRAWS,
        rng=np.random.RandomState(SEED),
        distance_column="placebo_distance",
        bw_l=bandwidth, bw_r=bandwidth, return_structures=True,
    )
    results, exact_statistics, observed_statistics = output[0], output[5], output[6]
    exact_compound = separately_pooled_exact_pvalues(
        exact_statistics, observed_statistics
    )
    results["exact_compound_p"] = results["placebo_id"].map(exact_compound)
    for column in ["local_p", "compound_p", "exact_compound_p"]:
        assert results[column].notna().all()
        assert (results[column] > 0).all()
    return results


def rejection_curve(results):
    return pd.DataFrame([{
        "alpha": level,
        "SL p-value": sl_procedure(results["local_p"].to_numpy(), level)[1],
        "SL Density-Compound": sl_procedure(
            results["compound_p"].to_numpy(), level
        )[1],
        "SL Compound": sl_procedure(
            results["exact_compound_p"].to_numpy(), level
        )[1],
    } for level in ALPHA_GRID])


In [ ]:
bandwidth_placebos = {}
bandwidth_samples = {}
bandwidth_results = {}
bandwidth_curves = {}
summary_rows = []
for bandwidth in BANDWIDTHS:
    retained, sample = filter_midpoint_placebo_design(
        midpoint_placebo_data, bandwidth=bandwidth,
        minimum_per_side=MINIMUM_SIDE_COUNT,
        maximum_total=MAXIMUM_TOTAL_COUNT,
    )
    results = compute_midpoint_results(sample, bandwidth)
    curve = rejection_curve(results)
    bandwidth_placebos[bandwidth] = retained
    bandwidth_samples[bandwidth] = sample
    bandwidth_results[bandwidth] = results
    bandwidth_curves[bandwidth] = curve
    row_at_alpha = curve.loc[np.isclose(curve["alpha"], ALPHA)].iloc[0]
    summary_rows.append({
        "bandwidth": bandwidth,
        "valid cohort-specific constructions": int(
            (year_gap_table["gap_width"]
             > 2 * bandwidth + NUMERICAL_TOLERANCE).sum()
        ),
        "testable pooled placebo constructions": len(results),
        "towns": sample["town"].nunique(),
        "median contributing cohorts": retained["contributing_years"].median(),
        "SL p-value": int(row_at_alpha["SL p-value"]),
        "SL Density-Compound": int(row_at_alpha["SL Density-Compound"]),
        "SL Compound": int(row_at_alpha["SL Compound"]),
        "minimum p-value": results["local_p"].min(),
        "minimum Density-Compound": results["compound_p"].min(),
        "minimum Compound": results["exact_compound_p"].min(),
    })
bandwidth_summary = pd.DataFrame(summary_rows)
display(bandwidth_summary)

fig, axes = plt.subplots(3, 3, figsize=(13.5, 11.2), sharex=True)
for ax, bandwidth in zip(axes.flat, BANDWIDTHS):
    curve = bandwidth_curves[bandwidth]
    ax.plot(curve["alpha"], curve["SL p-value"], color="#4C72B0",
            linewidth=2, label=r"SL ($p$-values)")
    ax.plot(curve["alpha"], curve["SL Density-Compound"],
            color="#C44E52", linewidth=2.2,
            label="SL (Density-Compound)")
    ax.plot(curve["alpha"], curve["SL Compound"], color="green",
            linewidth=2, label="SL (Compound)")
    ax.set_title(rf"$h={bandwidth:.2f}$ "
                 rf"($m={len(bandwidth_results[bandwidth])}$)")
    maximum = int(curve.iloc[:, 1:].to_numpy().max())
    if maximum == 0:
        ax.set_ylim(-1, 1)
        ax.set_yticks([-1, 0, 1])
    else:
        ax.set_ylim(bottom=-0.25)
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.grid(True, linestyle=":", alpha=0.4, color="#cccccc")
for ax in axes[:, 0]:
    ax.set_ylabel("Rejection Count")
for ax in axes[-1, :]:
    ax.set_xlabel(r"Nominal level $\alpha$")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, ncol=3, loc="upper center",
           bbox_to_anchor=(0.5, 0.99))
fig.tight_layout(rect=[0, 0, 1, 0.93])
_ = save_figure(fig, "rd_placebo_rejection_curves")
display(fig)
plt.close(fig)


## P-value histograms by bandwidth

Each panel shows the three p-value distributions for the corresponding family.

In [ ]:
histogram_series = [
    ("local_p", "#4C72B0", r"$p$-value"),
    ("compound_p", "#C44E52", "Density-Compound $p$-value"),
    ("exact_compound_p", "green", "Compound $p$-value"),
]
bins = np.linspace(0, 1, 21)
fig, axes = plt.subplots(3, 3, figsize=(13.5, 10.8), sharex=True)
for ax, bandwidth in zip(axes.flat, BANDWIDTHS):
    results = bandwidth_results[bandwidth]
    for column, color, _ in histogram_series:
        ax.hist(results[column], bins=bins, color=color, alpha=0.35,
                edgecolor="white", linewidth=0.6)
        ax.hist(results[column], bins=bins, histtype="step",
                color=color, linewidth=1.5)
    ax.set_title(rf"$h={bandwidth:.2f}$ ($m={len(results)}$)")
    ax.grid(True, axis="y", alpha=0.2)
for ax in axes[:, 0]:
    ax.set_ylabel("Count")
for ax in axes[-1, :]:
    ax.set_xlabel(r"$p$-value")
histogram_legend = [
    Line2D([0], [0], color=color, linewidth=2, label=label)
    for _, color, label in histogram_series
]
fig.legend(handles=histogram_legend, frameon=False, ncol=3,
           loc="upper center", bbox_to_anchor=(0.5, 0.99))
fig.tight_layout(rect=[0, 0, 1, 0.93])
_ = save_figure(fig, "rd_placebo_pvalue_histograms")
display(fig)
plt.close(fig)

## Rejection diagnostics

This section checks the ordinary-p rejection at $h=0.01$ and $\alpha=0.3$.
The display window is $0.5$; the fitted observations remain between the
neighboring real cutoffs. Figures are displayed but not saved.

In [ ]:
DIAGNOSTIC_ALPHA = 0.3
DIAGNOSTIC_BANDWIDTH = 0.01

diagnostic_results = bandwidth_results[DIAGNOSTIC_BANDWIDTH]
rejected_positions, _ = sl_procedure(
    diagnostic_results["local_p"].to_numpy(), DIAGNOSTIC_ALPHA
)
diagnostic_figures = diagnostic_results.iloc[
    np.asarray(rejected_positions, dtype=int)
][["placebo_id", "local_p", "compound_p",
   "exact_compound_p"]].copy()
diagnostic_figures.insert(0, "bandwidth", DIAGNOSTIC_BANDWIDTH)
diagnostic_figures.insert(2, "methods", "Ordinary")
diagnostic_figures = diagnostic_figures.reset_index(drop=True)
display(diagnostic_figures)


def placebo_diagnostic_data(sample):
    """Return display background and uncontaminated fit observations."""
    contributing_constructions = sample[
        ["placebo_id", "town", "Y", "z_low", "z_high"]
    ].drop_duplicates()
    background = midpoint_placebo_data.merge(
        contributing_constructions,
        on=["placebo_id", "town", "Y", "z_low", "z_high"],
        how="inner", validate="many_to_one",
    )
    background = background[
        (background["placebo_distance"].abs() > NUMERICAL_TOLERANCE)
        & (background["placebo_distance"].abs()
           <= LOCAL_RD_DISPLAY_WINDOW + NUMERICAL_TOLERANCE)
    ].copy()
    fit_data = background[
        background["placebo_distance"].abs()
        < background["gap_width"] / 2 - NUMERICAL_TOLERANCE
    ].copy()
    fit_bandwidth = min(
        BASELINE_BANDWIDTH,
        fit_data["gap_width"].min() / 2 - NUMERICAL_TOLERANCE,
    )
    return background, fit_data, fit_bandwidth
extra_rows = []
for _, rejection in diagnostic_figures.iterrows():
    bandwidth = float(rejection["bandwidth"])
    sample = bandwidth_samples[bandwidth]
    sample = sample[sample["placebo_id"] == rejection["placebo_id"]]
    left = sample[sample["placebo_distance"] < 0]
    right = sample[sample["placebo_distance"] > 0]
    _, fit_data, fit_bandwidth = placebo_diagnostic_data(sample)
    try:
        robust = rdrobust(
            y=fit_data["outcome"], x=fit_data["placebo_distance"],
            cluster=fit_data["sid2"], vce="cr3",
            h=fit_bandwidth,
        )
        robust_p = robust.pv["P>|z|"].iloc[2]
    except Exception:
        # rdrobust can be unidentified in these deliberately tiny, discrete
        # local samples; this annotation is diagnostic only.
        robust_p = np.nan
    extra_rows.append({
        "bandwidth": bandwidth, "placebo_id": rejection["placebo_id"],
        "n_left": len(left), "n_right": len(right),
        "contributing_years": sample["Y"].nunique(),
        "right_minus_left": right["outcome"].mean() - left["outcome"].mean(),
        "rdrobust_fit_n": len(fit_data),
        "rdrobust_fit_bandwidth": fit_bandwidth,
        "rdrobust_p": robust_p,
    })
if extra_rows:
    diagnostic_figures = diagnostic_figures.merge(
        pd.DataFrame(extra_rows), on=["bandwidth", "placebo_id"],
        how="left", validate="one_to_one"
    )

if len(diagnostic_figures) == 0:
    print(
        f"No ordinary-p rejection at h={DIAGNOSTIC_BANDWIDTH:.2f} "
        f"and alpha={DIAGNOSTIC_ALPHA:.1f}."
    )
else:
    columns = min(2, len(diagnostic_figures))
    rows = int(np.ceil(len(diagnostic_figures) / columns))
    fig, axes = plt.subplots(
        rows, columns, figsize=(8 * columns, 5.8 * rows + 1.2),
                             squeeze=False)
    for ax, (_, rejection) in zip(axes.flat, diagnostic_figures.iterrows()):
        bandwidth = float(rejection["bandwidth"])
        sample = bandwidth_samples[bandwidth]
        sample = sample[sample["placebo_id"] == rejection["placebo_id"]]
        left = sample[sample["placebo_distance"] < 0]
        right = sample[sample["placebo_distance"] > 0]
        background, fit_data, fit_bandwidth = placebo_diagnostic_data(sample)
        fit_window = fit_data[
            fit_data["placebo_distance"].abs()
            <= fit_bandwidth + NUMERICAL_TOLERANCE
        ]
        ax.scatter(
            background["placebo_distance"], background["outcome"],
            color="#bdc3c7", s=15, alpha=0.35, edgecolors="none",
            rasterized=True, label=rf"Background ($|x|\leq{LOCAL_RD_DISPLAY_WINDOW}$)",
        )
        ax.scatter(left["placebo_distance"], left["outcome"],
                   color="#4C72B0", s=28, alpha=0.7, edgecolors="none",
                   label=rf"Retained left ($h={bandwidth:.2f}$)")
        ax.scatter(right["placebo_distance"], right["outcome"],
                   color="#C44E52", s=28, alpha=0.7, edgecolors="none",
                   label=rf"Retained right ($h={bandwidth:.2f}$)")
        fit_left = fit_window[
            fit_window["placebo_distance"] < 0
        ]
        fit_right = fit_window[
            fit_window["placebo_distance"] > 0
        ]
        for points, start, stop in [
            (fit_left, -fit_bandwidth, -0.0001),
            (fit_right, 0.0001, fit_bandwidth),
        ]:
            x = points["placebo_distance"].to_numpy()
            y = points["outcome"].to_numpy()
            grid = np.linspace(start, stop, 100)
            if np.unique(x).size >= 2:
                kernel_weight = np.clip(
                    1 - np.abs(x) / fit_bandwidth, 0, None
                )
                fit = np.polyval(
                    np.polyfit(x, y, 1, w=np.sqrt(kernel_weight)), grid
                )
            else:
                fit = np.repeat(np.mean(y), len(grid))
            ax.plot(grid, fit, color="#2c3e50", linewidth=1.6)
        ax.axvspan(-bandwidth, bandwidth, color=PLACEBO_COLOR, alpha=0.07)
        ax.axvline(0, color="#7f8c8d", linestyle="--", linewidth=1)
        for half_gap in np.sort(sample["gap_width"].unique() / 2):
            ax.axvline(-half_gap, color="#95a5a6", linestyle=":",
                       linewidth=1, alpha=0.7)
            ax.axvline(half_gap, color="#95a5a6", linestyle=":",
                       linewidth=1, alpha=0.7)
        robust_label = (f"{rejection['rdrobust_p']:.6f}"
                        if pd.notna(rejection["rdrobust_p"])
                        else "not estimable")
        safe_id = str(rejection["placebo_id"]).replace("_", r"\_")
        ax.set_title(
            rf"$h={bandwidth:.2f}$, $\mathrm{{{safe_id}}}$" + "\n" +
            f"Rejected by: {rejection['methods']}" + "\n" +
            rf"$p^{{\rm perm}}={rejection['local_p']:.6f}$ | " +
            rf"$p^{{\rm DCP}}={rejection['compound_p']:.6f}$ | " +
            rf"$p^{{\rm compound}}={rejection['exact_compound_p']:.6f}$ | " +
            rf"$p^{{\rm rdrobust}}={robust_label}$ " +
            rf"($h_{{\rm fit}}={rejection['rdrobust_fit_bandwidth']:.2f}$)",
            fontsize=9.5
        )
        ax.set_xlabel("Distance to Cohort-specific Placebo Cutoff")
        ax.set_xlim(-LOCAL_RD_DISPLAY_WINDOW, LOCAL_RD_DISPLAY_WINDOW)
        ax.set_ylabel("Baccalaureate Grade")
        ax.grid(True, linestyle=":", alpha=0.3, color="#cccccc")
    for ax in axes.flat[len(diagnostic_figures):]:
        ax.set_visible(False)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, ncol=3, loc="upper center",
               bbox_to_anchor=(0.5, 0.995))
    fig.tight_layout(rect=[0, 0, 1, 0.80])
    display(fig)
    plt.close(fig)

    for _, rejection in diagnostic_figures.iterrows():
        bandwidth = float(rejection["bandwidth"])
        sample = bandwidth_samples[bandwidth]
        sample = sample[sample["placebo_id"] == rejection["placebo_id"]]
        years = sorted(sample["Y"].unique())
        town = int(sample["town"].iloc[0])
        full_background = midpoint_placebo_data[
            midpoint_placebo_data["placebo_id"] == rejection["placebo_id"]
        ]
        fig, axes = plt.subplots(1, len(years), figsize=(5 * len(years), 4),
                                 squeeze=False, sharey=True)
        for ax, year in zip(axes.flat, years):
            cohort = sample[sample["Y"] == year]
            cohort_background = full_background[full_background["Y"] == year]
            gap_width = float(cohort["gap_width"].iloc[0])
            gap_info = year_gap_table.loc[
                (year_gap_table["town"] == town)
                & (year_gap_table["Y"] == year)
                & (year_gap_table["gap_rank"] == cohort["gap_rank"].iloc[0])
            ].iloc[0]
            midpoint = float(gap_info["midpoint_position"])
            ax.scatter(cohort_background["placebo_distance"],
                       cohort_background["outcome"], color="#bdc3c7", s=10,
                       alpha=0.22, edgecolors="none", rasterized=True)
            ax.scatter(cohort["placebo_distance"], cohort["outcome"],
                       color=YEAR_COLORS.get(year, "#777777"), s=25,
                       alpha=0.7, edgecolors="none")
            ax.axvspan(-bandwidth, bandwidth, color=PLACEBO_COLOR, alpha=0.1)
            ax.axvline(0, color=PLACEBO_COLOR, linestyle="--", linewidth=1.5)
            for real_cutoff in cutoff_table.loc[
                (cutoff_table["town"] == town)
                & (cutoff_table["Y"] == year), "cutoff_position"
            ]:
                ax.axvline(real_cutoff - midpoint,
                           color=YEAR_COLORS.get(year, "#777777"),
                           linewidth=0.8, alpha=0.35)
            ax.axvline(-gap_width / 2, color="#333333", linestyle=":",
                       linewidth=1.3)
            ax.axvline(gap_width / 2, color="#333333", linestyle=":",
                       linewidth=1.3)
            ax.set_title(rf"Town {town}, cohort $Y={year}$" + "\n" +
                         rf"gap width $={gap_width:.2f}$, $h={bandwidth:.2f}$")
            ax.set_xlabel("Distance to Cohort-specific Placebo Cutoff")
            ax.grid(True, linestyle=":", alpha=0.3, color="#cccccc")
        axes.flat[0].set_ylabel("Baccalaureate Grade")
        fig.tight_layout()
        display(fig)
        plt.close(fig)